# Steering — Base models (Table 14)

DiffAware **generation-time steering** on base models: scale the attention
edge item -> B (the culturally associated identity) in the binding heads by a
factor alpha during greedy generation, then parse the generated answer.

Reproduces **Table 14** of the paper: neq accuracy (cultural questions,
correct = culturally associated option) and eq accuracy (neutral questions,
correct = option (c) "equally likely") as a function of alpha, per base model.

Note: the evaluation here is a **simplified generation accuracy** (parse the
generated letter with `map_to_abc`), which differs by design from the
instruct-side steering notebook (logit-based S-scores). This methodological
difference is intentional.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

import pickle
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

from common import config

CFG = config.init(MODEL_KEY, "base")

SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN
MC = config.MC
MAX_NEW_TOKENS = config.MAX_NEW_TOKENS
HEADS = config.HEADS
ANSWER_START = config.ANSWER_START
OUTPUT_DIR = config.OUTPUT_DIR

from common.text_parsers import norm_identity, extract_options, parse_meta
from common.base.data import load_n4
# NB: extract_neutral_item must come from common.base.prompts (the steering
# version), not common.base.data (the dataset-construction version). The
# two differ, and importing the wrong one would silently change behavior
# here.
from common.base.prompts import (TEMPLATES, REFUSALS, clean_response,
                                 map_to_abc, extract_neutral_item,
                                 find_identity_and_item_positions)
from common.base.steering import generate_steered

## Data and model loading

In [ ]:
cultural_items, neutral_items = load_n4(DATA_DIR)

In [ ]:
print(f"Loading {CFG['path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG["path"], trust_remote_code=True, token=HF_TOKEN
)
model = AutoModelForCausalLM.from_pretrained(
    CFG["path"], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
first_device = next(model.parameters()).device

n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
n_kv = getattr(model.config, "num_key_value_heads", n_heads)
softcap_val = getattr(model.config, "attn_logit_softcapping", None)

print(f"  {n_layers} layers, {n_heads} Q-heads, {n_kv} KV-heads (GQA group={n_heads // n_kv})")
print(f"  Softcapping: {softcap_val}")
print(f"  Search space: {n_layers * n_heads} (layer, head) pairs")

# Register runtime singletons so common helpers can access them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

## neq / eq generation prompts and token positions

In [ ]:
import transformers
import io, contextlib

# -- Build neq/eq data from original N4 --
neq_data, eq_data = [], []

for item in cultural_items:
    q, ans, meta = item
    try:
        qtype, c_item, in_groups, out_groups = parse_meta(meta)
    except Exception:
        continue
    oa, ob = extract_options(q)
    if oa is None:
        continue
    assoc_pos = None
    for g in in_groups:
        if norm_identity(g) == norm_identity(oa):
            assoc_pos = "a"; break
        elif norm_identity(g) == norm_identity(ob):
            assoc_pos = "b"; break
    if assoc_pos is None:
        continue
    neq_data.append({
        "prompt": q + "\n\n" + MC, "ans": int(ans), "item": c_item,
        "assoc_pos": assoc_pos, "oa": oa, "ob": ob,
    })

eq_skipped = 0
for item in neutral_items:
    q, ans, meta = item
    try:
        qtype, c_item, in_groups, out_groups = parse_meta(meta)
    except Exception:
        eq_skipped += 1
        continue
    oa, ob = extract_options(q)
    if oa is None:
        eq_skipped += 1
        continue
    assoc_pos = None
    for g in in_groups:
        if norm_identity(g) == norm_identity(oa):
            assoc_pos = "a"; break
        elif norm_identity(g) == norm_identity(ob):
            assoc_pos = "b"; break
    if assoc_pos is None:
        eq_skipped += 1
        continue
    neutral_item = extract_neutral_item(q, qtype)
    if neutral_item is None:
        eq_skipped += 1
        continue
    eq_data.append({
        "prompt": q + "\n\n" + MC, "ans": 2, "item": neutral_item,
        "assoc_pos": assoc_pos, "oa": oa, "ob": ob,
    })

print(f"  neq: {len(neq_data)}, eq: {len(eq_data)} (eq skipped: {eq_skipped})")

# -- Positions for generation prompts --
neq_positions, eq_positions = [], []
for lst, pos_out in [(neq_data, neq_positions), (eq_data, eq_positions)]:
    for d in lst:
        fmt = d["prompt"] + "\n" + ANSWER_START
        pos = find_identity_and_item_positions(
            tokenizer, fmt, d["prompt"], d["item"], d["assoc_pos"])
        pos_out.append(pos)

print(f"  neq positions: {sum(1 for p in neq_positions if p)}/{len(neq_data)} valid")
print(f"  eq  positions: {sum(1 for p in eq_positions if p)}/{len(eq_data)} valid")

## DiffAware steering across alpha values

In [ ]:
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

# -- Generation pipeline --
pipe = transformers.pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device_map="auto",
)
if pipe.model.config.pad_token_id is None:
    pipe.model.config.pad_token_id = pipe.model.config.eos_token_id

# -- Run steering --
DA_ALPHAS = [0.0, 1.0, 2.0, 3.0, 5.0, 7.0]
da_results = {}

print(f"DiffAware steering ({len(DA_ALPHAS)} alpha values)")
print(f"  Heads: {HEADS}")
print(f"  max_new_tokens: {MAX_NEW_TOKENS}\n")

for alpha in DA_ALPHAS:
    print(f"  alpha = {alpha:.1f} ...")

    with contextlib.redirect_stdout(io.StringIO()):
        neq_gen = generate_steered(pipe, neq_data, neq_positions, HEADS, alpha,
                                   max_new_tokens=MAX_NEW_TOKENS)
        eq_gen = generate_steered(pipe, eq_data, eq_positions, HEADS, alpha,
                                  max_new_tokens=MAX_NEW_TOKENS)

    neq_correct = sum(1 for (p, _), d in zip(neq_gen, neq_data) if p == d["ans"])
    eq_correct = sum(1 for (p, _), d in zip(eq_gen, eq_data) if p == 2)
    neq_acc = neq_correct / len(neq_data)
    eq_acc = eq_correct / len(eq_data)

    A = neq_correct
    DE = len(eq_data) - eq_correct
    cxt = A / (A + DE) if (A + DE) > 0 else 0

    da_results[alpha] = {
        "neq_acc": neq_acc, "eq_acc": eq_acc,
        "overall": (neq_correct + eq_correct) / (len(neq_data) + len(eq_data)),
        "cxt_aware": cxt,
    }

    print(f"    neq acc: {neq_acc:.3f} ({neq_correct}/{len(neq_data)})")
    print(f"    eq  acc: {eq_acc:.3f} ({eq_correct}/{len(eq_data)})")
    print(f"    CxtAware: {cxt:.3f}")
    torch.cuda.empty_cache()

In [ ]:
# Persistence: per-alpha steering accuracies.
with open(OUTPUT_DIR / "steering_da_results.pkl", "wb") as f:
    pickle.dump(da_results, f)
print(f"Saved {OUTPUT_DIR / 'steering_da_results.pkl'}")

### Steering results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

alphas_da = sorted(da_results.keys())
neq_accs = [da_results[a]["neq_acc"] for a in alphas_da]
eq_accs = [da_results[a]["eq_acc"] for a in alphas_da]

ax.plot(alphas_da, neq_accs, "s-", color="#d62728", linewidth=2, markersize=8, label="neq acc (cultural)")
ax.plot(alphas_da, eq_accs, "o-", color="#1f77b4", linewidth=2, markersize=8, label="eq acc (neutral)")
ax.axvline(x=1.0, color="gray", ls=":", alpha=0.3)
ax.set_xlabel("alpha (attention scale factor)")
ax.set_ylabel("Accuracy")
ax.set_title(f"{MODEL_KEY.upper()} BASE: DiffAware steering")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "steering_da_plot.png", dpi=150, bbox_inches="tight")  # persist figure
plt.show()